# 00 — Sanity checks: is the reversible backward *really* exact and memory-free?

Before training anything, this notebook verifies the three claims that everything else rests on:

1. **Reversibility is exact.** Running the layers forward then `inverse()` returns the input (error ≈ float round-off).
2. **The memory-free backward gives the same gradients** as ordinary autograd on the same architecture.
3. **Activation memory is constant in depth** for reversible models and linear for the baseline (paper Fig. 3).

We also measure the price: how much slower is one training step when activations are recomputed instead of stored?

In [1]:
# @title Setup — clone repo (if needed), install deps, detect GPU
import os, sys, subprocess, json, time, math
REPO_URL = "https://github.com/YOUR_GITHUB_USER/reversible-llm-poc.git"   # <-- edit after you push

if not os.path.exists("src/revllm.py"):
    if os.path.exists("../src/revllm.py"):
        os.chdir("..")
    else:
        subprocess.run(["git", "clone", "-q", REPO_URL, "reversible-llm-poc"], check=True)
        os.chdir("reversible-llm-poc")
sys.path.insert(0, os.path.abspath("src"))
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "tiktoken", "datasets", "matplotlib"], check=False)

import torch
from revllm import Config, GPT, SavedTensorMeter
from data import prepare_tinystories, prepare_synthetic, TokenStream
import train as T

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# SMOKE mode = tiny synthetic run that finishes in ~1 min on CPU. Auto-enabled when there is no GPU.
SMOKE = os.environ.get("SMOKE", "0") == "1" or DEVICE == "cpu"
print("device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu", "| SMOKE mode:", SMOKE)
os.makedirs("results", exist_ok=True)


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


device: cpu | SMOKE mode: True


In [2]:
# @title Experiment configuration (shared by all notebooks)
if SMOKE:
    MODEL  = dict(vocab_size=512, block_size=64, n_layer=4, n_embd=128, n_head=4)
    TOKENS = 200_000          # token budget
    BATCH  = 16               # the fixed batch size for notebooks 01/02
    LR     = 2e-3
    LOG    = dict(eval_every=100, eval_iters=5, log_every=50)
else:
    # ~20.8M parameters (12.9M in the tied GPT-2 embedding, 7.9M in 10 transformer blocks of width 256)
    MODEL  = dict(vocab_size=50257, block_size=256, n_layer=10, n_embd=256, n_head=4)
    TOKENS = 50_000_000
    BATCH  = 32               # 32 x 256 = 8,192 tokens / step  -> ~6,100 steps for 50M tokens
    LR     = 6e-4
    LOG    = dict(eval_every=250, eval_iters=20, log_every=50)

def get_data():
    if SMOKE:
        return prepare_synthetic("data/synthetic", 2_000_000, 200_000, vocab=MODEL["vocab_size"])
    return prepare_tinystories("data/tinystories", n_train_tokens=55_000_000, n_val_tokens=2_000_000)

def gpu_table(rows, headers):
    w = [max(len(str(r[i])) for r in [headers] + rows) for i in range(len(headers))]
    line = lambda r: "| " + " | ".join(str(c).ljust(w[i]) for i, c in enumerate(r)) + " |"
    print(line(headers)); print("|" + "|".join("-" * (x + 2) for x in w) + "|")
    for r in rows: print(line(r))

import matplotlib.pyplot as plt
def plot_runs(results, key="curve", title="training loss", smooth=25):
    plt.figure(figsize=(8, 4.5))
    for r in results:
        c = r[key]
        if not c: continue
        xs = [p[1] / 1e6 for p in c]; ys = [p[2] for p in c]
        if key == "curve" and smooth > 1 and len(ys) > smooth:
            ys = [sum(ys[max(0, i - smooth):i + 1]) / len(ys[max(0, i - smooth):i + 1]) for i in range(len(ys))]
        plt.plot(xs, ys, label=f"{r['run_name']}  (final {ys[-1]:.3f})")
    plt.xlabel("tokens seen (M)"); plt.ylabel("cross-entropy (nats)"); plt.title(title); plt.legend(); plt.grid(alpha=.3)
    plt.show()


## 1 & 2 — exact inverse and identical gradients (all three reversible variants)

In [3]:
import copy
torch.manual_seed(0)
small = dict(MODEL, n_layer=4)
x = torch.randint(0, small["vocab_size"], (2, small["block_size"]), device=DEVICE)
y = torch.randint(0, small["vocab_size"], (2, small["block_size"]), device=DEVICE)
rows = []
for mode in ["midpoint", "leapfrog", "hamiltonian"]:
    m_rev = GPT(Config(mode=mode, rev_backprop=True, **small)).to(DEVICE)
    m_std = copy.deepcopy(m_rev); m_std.layers.rev_backprop = False
    l_rev = m_rev(x, y); l_rev.backward()
    l_std = m_std(x, y); l_std.backward()
    gdiff = max((a.grad - b.grad).abs().max().item() for a, b in zip(m_rev.parameters(), m_std.parameters()) if a.grad is not None and b.grad is not None)
    with torch.no_grad():
        p0 = m_rev.wte(x) + m_rev.wpe(torch.arange(x.shape[1], device=DEVICE))
        y1, y2 = m_rev.layers(p0, p0)
        r1, r2 = m_rev.layers.inverse(y1, y2)
        recon = max((r1 - p0).abs().max().item(), (r2 - p0).abs().max().item())
    rows.append([mode, f"{l_rev.item():.6f}", f"{l_std.item():.6f}", f"{gdiff:.1e}", f"{recon:.1e}"])
gpu_table(rows, ["variant", "loss (rev backprop)", "loss (std autograd)", "max |grad diff|", "reconstruction err"])
print("\n(gradient differences of ~1e-8 are float32 round-off; anything > 1e-4 would indicate a bug)")

| variant     | loss (rev backprop) | loss (std autograd) | max |grad diff| | reconstruction err |
|-------------|---------------------|---------------------|-----------------|--------------------|
| midpoint    | 6.249822            | 6.249822            | 6.0e-08         | 2.0e-07            |
| leapfrog    | 6.251880            | 6.251880            | 3.9e-08         | 1.9e-07            |
| hamiltonian | 6.270794            | 6.270794            | 1.3e-08         | 7.1e-08            |

(gradient differences of ~1e-8 are float32 round-off; anything > 1e-4 would indicate a bug)


## 3 — activation memory vs depth

`SavedTensorMeter` counts the bytes autograd stashes for the backward pass — a platform-independent measure of activation memory. On a GPU we also read `torch.cuda.max_memory_allocated()` for a full forward+backward. Depth grows, batch stays fixed.

In [4]:
depths = [2, 4, 8, 16, 32] if not SMOKE else [2, 4, 8]
B, Tn = (8, MODEL["block_size"])
rows, saved, peak = [], {}, {}
for L in depths:
    for mode, rev in [("baseline", True), ("midpoint", False), ("midpoint", True)]:
        name = mode + ("" if mode == "baseline" else ("/rev" if rev else "/no-rev"))
        cfg = Config(mode=mode, rev_backprop=rev, head_chunk=0, **dict(MODEL, n_layer=L))
        m = GPT(cfg).to(DEVICE)
        xb = torch.randint(0, cfg.vocab_size, (B, Tn), device=DEVICE)
        if DEVICE == "cuda": torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
        with SavedTensorMeter() as sm:
            loss = m(xb, xb)
        loss.backward()
        if DEVICE == "cuda": torch.cuda.synchronize(); pk = torch.cuda.max_memory_allocated() / 1e9
        else: pk = float("nan")
        saved.setdefault(name, []).append(sm.bytes / 1e6); peak.setdefault(name, []).append(pk)
        rows.append([L, name, f"{sm.bytes/1e6:.1f}", f"{pk:.2f}" if pk == pk else "n/a"])
        del m, loss, xb
        if DEVICE == "cuda": torch.cuda.empty_cache()
gpu_table(rows, ["layers", "model", "saved-for-backward (MB)", "GPU peak fwd+bwd (GB)"])

fig, ax = plt.subplots(1, 2 if DEVICE == "cuda" else 1, figsize=(11, 4)); ax = ax if isinstance(ax, (list, tuple)) or hasattr(ax, "__len__") else [ax]
for name in saved:
    ax[0].plot(depths, saved[name], marker="o", label=name)
ax[0].set_xlabel("layers"); ax[0].set_ylabel("MB saved for backward"); ax[0].set_title("activation memory vs depth"); ax[0].legend(); ax[0].grid(alpha=.3)
if DEVICE == "cuda":
    for name in peak: ax[1].plot(depths, peak[name], marker="o", label=name)
    ax[1].set_xlabel("layers"); ax[1].set_ylabel("GB (peak, fwd+bwd, B=%d)" % B); ax[1].set_title("GPU peak memory vs depth"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.savefig("results/memory_vs_depth.png", dpi=120); plt.show()
json.dump({"depths": depths, "saved_MB": saved, "peak_GB": peak, "batch": B, "block": Tn, "device": DEVICE},
          open("results/memory_vs_depth.json", "w"), indent=1)

| layers | model           | saved-for-backward (MB) | GPU peak fwd+bwd (GB) |
|--------|-----------------|-------------------------|-----------------------|
| 2      | baseline        | 13.4                    | n/a                   |
| 2      | midpoint/no-rev | 13.4                    | n/a                   |
| 2      | midpoint/rev    | 3.4                     | n/a                   |
| 4      | baseline        | 23.9                    | n/a                   |
| 4      | midpoint/no-rev | 23.9                    | n/a                   |
| 4      | midpoint/rev    | 3.4                     | n/a                   |
| 8      | baseline        | 45.0                    | n/a                   |
| 8      | midpoint/no-rev | 45.0                    | n/a                   |
| 8      | midpoint/rev    | 3.4                     | n/a                   |


## 4 — the price of not storing activations: step-time overhead

Same midpoint architecture, one full training step, with and without memory-free backward. The paper says recomputation "typically adds only 30–50% overhead" on GPU (§2.2).

In [5]:
def time_step(cfg, B, n=10):
    ctx, scaler, _ = T._amp(DEVICE)
    m = GPT(cfg).to(DEVICE); opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
    xb = torch.randint(0, cfg.vocab_size, (B, cfg.block_size), device=DEVICE)
    ts = []
    for i in range(n + 3):
        if DEVICE == "cuda": torch.cuda.synchronize()
        t0 = time.time()
        with ctx: loss = m(xb, xb)
        loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
        if DEVICE == "cuda": torch.cuda.synchronize()
        if i >= 3: ts.append(time.time() - t0)
    return sum(ts) / len(ts)

Bt = 8 if SMOKE else 32
rows = []
base = time_step(Config(mode="baseline", **MODEL), Bt)
for mode in ["midpoint", "leapfrog", "hamiltonian"]:
    t_std = time_step(Config(mode=mode, rev_backprop=False, **MODEL), Bt)
    t_rev = time_step(Config(mode=mode, rev_backprop=True, **MODEL), Bt)
    rows.append([mode, f"{t_std*1000:.0f}", f"{t_rev*1000:.0f}", f"+{(t_rev/t_std-1)*100:.0f}%", f"{t_rev/base:.2f}x"])
gpu_table(rows, ["variant", "step ms (std autograd)", "step ms (rev backprop)", "overhead", "vs baseline step"])
print(f"\nbaseline step: {base*1000:.0f} ms at batch {Bt}")

| variant     | step ms (std autograd) | step ms (rev backprop) | overhead | vs baseline step |
|-------------|------------------------|------------------------|----------|------------------|
| midpoint    | 18                     | 31                     | +70%     | 1.70x            |
| leapfrog    | 18                     | 31                     | +68%     | 1.71x            |
| hamiltonian | 18                     | 30                     | +67%     | 1.68x            |

baseline step: 18 ms at batch 8
